# Drive Failure Prediction from 90-Day SMART WindowsA refactor of `Dataset_backblaze_Comparison.ipynb` from **row-level** to **window-level**classification.| | row-level notebook | this notebook ||---|---|---|| a sample is | one drive-day | one drive's **90 consecutive days** || X shape | `(batch, 129)` hand-built aggregates | `(batch, 90, num_features)` raw daily sequence || label | will this drive fail in 10 days? | does a failure fall **inside this window**? || model | MLP over aggregates | dilated 1D-CNN / GRU / Transformer over the sequence || split | `GroupShuffleSplit` on `drive_id` | same — windows of one disk never cross splits |The trailing-window statistics the row-level pipeline computed by hand (`_d7`, `_mean90`,`_std90`, `_max90`) are gone: the sequence model is handed the 90 raw daily readings andderives whatever summary it needs.Everything is implemented in [`scripts/backblaze_window_pipeline.py`](scripts/backblaze_window_pipeline.py);this notebook drives it and draws the results.**Dataset:** Backblaze quarterly releases -- Seagate drives only.

## 1. Setup

In [ ]:
import os, sys, json, time, gc
sys.path.insert(0, "scripts")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve

from backblaze_window_pipeline import (
    DataConfig, TrainConfig, SMART_RAW_COLUMNS, LOG1P_COLUMNS,
    load_backblaze_frame, make_synthetic_frame,
    build_window_bundle, build_loaders, run_sanity_checks, inspect_windows,
    build_model, count_parameters, train_model, evaluate_final, evaluate_loader,
    make_criterion, compute_metrics, save_artifacts, set_seed,
)

%matplotlib inline

SEED = 42
set_seed(SEED)
torch.set_num_threads(os.cpu_count() or 4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- run configuration -------------------------------------------------------
# Sized for a CPU run over one quarter. On a GPU, raise HIDDEN_DIM to 64 and
# MAX_EPOCHS to 20-30, and try ARCH = "transformer".
ARCH = "cnn"          # cnn | gru | transformer
HIDDEN_DIM = 48
BATCH_SIZE = 512
MAX_EPOCHS = 10
PATIENCE = 3
SAMPLER = "balanced"     # batch resampling; see section 4

# Categorical slots 1-3 of the validated default palette (all-pairs safe under CVD).
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "grid.alpha": 0.7,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK_2, "ytick.color": INK_2, "font.size": 10,
    "axes.titlesize": 11, "legend.frameon": False, "lines.linewidth": 2,
})

print(f"torch {torch.__version__} | device {device} | threads {torch.get_num_threads()}")

## 2. DataBackblaze publishes one CSV per day; `load_backblaze_frame` downloads a quarter, keepsthe Seagate drives and the nine SMART attributes the row-level notebook settled on, andcaches the result as parquet. The bulk data is staged **outside** the project folder —the extracted quarter is several GB and this project lives in a synced OneDrivedirectory.Set `DATA_DIR` to wherever you want the download to land. If the cache is missing thenotebook falls back to a synthetic frame so it still runs end to end.

In [ ]:
DATA_DIR = os.environ.get("BACKBLAZE_DATA_DIR", r"C:\Users\User\AppData\Local\Temp\backblaze_data")
CACHE = os.path.join(DATA_DIR, "backblaze_seagate_h1_2023.parquet")
QUARTERS = ["Q1_2023", "Q2_2023"]
ARTIFACT_PREFIX = os.path.join("models", "backblaze_window")
FEATURE_COLUMNS = list(SMART_RAW_COLUMNS)
LOG1P_COLS = list(LOG1P_COLUMNS)

t0 = time.time()
if os.path.exists(CACHE):
    raw_df = pd.read_parquet(CACHE)
    source = f"cache {os.path.basename(CACHE)}"
else:
    try:
        raw_df = load_backblaze_frame(QUARTERS, model_prefix_filter="ST",
                                      cache=CACHE, data_dir=DATA_DIR)
        source = "Backblaze download"
    except Exception as exc:                       # offline / no disk space
        print(f"falling back to synthetic data: {type(exc).__name__}: {exc}")
        raw_df = make_synthetic_frame(n_drives=3000, seed=SEED)
        source = "SYNTHETIC (not real telemetry)"

# 20M drive ids as Python strings cost about a gigabyte; as category codes, ~80 MB.
raw_df["id"] = raw_df["id"].astype("category")
raw_df["model"] = raw_df["model"].astype("category")

print(f"source          : {source}  ({time.time() - t0:.0f}s)")
print(f"drive-days      : {len(raw_df):,}")
print(f"drives          : {raw_df['id'].nunique():,}")
print(f"drive models    : {raw_df['model'].nunique()}")
print(f"date range      : {raw_df['time'].min().date()} -> {raw_df['time'].max().date()}")
print(f"failure events  : {int(raw_df['failure'].sum()):,}")
print(f"drives that fail: {raw_df.loc[raw_df['failure'] == 1, 'id'].nunique():,}")
print(f"row-level positive rate: {100 * raw_df['failure'].mean():.5f}%")
raw_df.head()

## 3. Windowing`DriveSeriesStore` puts every drive on a **gap-free daily calendar**: a drive seen on the1st and the 5th gets rows for the 2nd–4th, forward-filled from the last real reading andflagged in an `observed` channel the model can see. Counters are `log1p`-compressed, and7-day differences are added as extra channels — 19 channels in all. (A 1-daydifference channel is available via `delta_lags=(1, 7)`; it is left out here because akernel-3 convolution computes it in its first layer anyway, and 20M rows x 9 extrafloat32 channels is 750 MB of RAM that this machine would rather spend elsewhere.)The label rule is the one thing that must be exactly right:> **y = 1** if a `failure == 1` event falls inside `[start, start + 89]`, else **0**.`failure` itself is held in a separate array and is never part of X.

In [ ]:
cfg = DataConfig(
    window_days=90,
    horizon_days=0,          # strict: the failure must land INSIDE the window
    stride_days=15,          # val/test enumerate a window every N days
    samples_per_drive=1,     # random training windows per drive per epoch
    train_positive_ratio=0.5,
    min_days=30,             # shorter drives are dropped; 30-89 days are left-padded
    delta_lags=(7,),         # channels: levels + 7-day deltas + the observed mask
    feature_columns=tuple(FEATURE_COLUMNS),
    log1p_columns=tuple(LOG1P_COLS),
    random_state=SEED,
)

bundle = build_window_bundle(raw_df, cfg)

# The store holds its own dense copy; the 20M-row frame is not needed again.
del raw_df
gc.collect()
print(f"\nX channels ({bundle.num_features}): {bundle.feature_names}")

## 4. Sanity checks — before a single gradient step`run_sanity_checks` prints, in order:1. **the leakage guard** — that `failure` is absent from X and that no channel is a copy   of it;2. **split integrity** — drive counts per split and the (empty) pairwise id overlaps;3. **window counts** and the positive rate per split;4. **representative windows** — drive id, start/end dates, tensor shape, label, how much   of the window is real vs. forward-filled vs. padded, and the actual values across the   90 days in both scaled and original units;5. **the first train and test batches** as they come out of the DataLoader — shape, label   distribution, value statistics, and a numeric slice.

In [ ]:
tcfg = TrainConfig(
    arch=ARCH, hidden_dim=HIDDEN_DIM, dropout=0.2,
    batch_size=BATCH_SIZE, lr=1e-3, max_epochs=MAX_EPOCHS, patience=PATIENCE,
    loss="bce_pos_weight", sampler=SAMPLER, balanced_pos_fraction=0.25,
    num_workers=0, seed=SEED,
)
loaders = build_loaders(bundle, tcfg, device)
run_sanity_checks(bundle, loaders, n_samples=3)

## 5. ModelA dilated 1D CNN over the window. Dilations 1→16 across five residual blocks give areceptive field of 125 days, so the classification head sees the whole window at once.Pooling concatenates mean, max and the final timestep: the level, the worst moment, andthe drive's most recent state.`GRUWindowClassifier` and `TransformerWindowClassifier` are drop-in alternatives with thesame `(B, 90, F) -> (B, 1)` contract.

In [ ]:
model = build_model(bundle.num_features, cfg.window_days, tcfg).to(device)
print(f"{tcfg.arch}: {count_parameters(model):,} trainable parameters\n")
print(model)

with torch.no_grad():
    probe = torch.zeros(4, cfg.window_days, bundle.num_features, device=device)
    print(f"\nforward: {tuple(probe.shape)} -> {tuple(model(probe).shape)}  (one logit per window)")

## 6. TrainingEarly stopping on **validation PR-AUC**, not loss or accuracy: at a positive rate of afraction of a percent, accuracy is meaningless and the loss moves with the imbalancecorrection rather than with the ranking that actually gets used.The imbalance is corrected **once** — a balanced sampler oversamples failure-carryingdrives, and `make_criterion` therefore drops `pos_weight` back to 1. Correcting it twiceis the mistake documented in `TOSHIBA_PIPELINE.md`.

In [ ]:
t0 = time.time()
model, history, best_val_pr_auc = train_model(model, loaders, bundle, tcfg, device)
print(f"\nbest validation PR-AUC {best_val_pr_auc:.4f} in {(time.time() - t0) / 60:.1f} min")

In [ ]:
epochs = np.arange(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history["train_loss"], color=BLUE, label="train")
axes[0].plot(epochs, history["val_loss"], color=ORANGE, label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BCE loss"); axes[0].legend()

axes[1].plot(epochs, history["val_pr_auc"], color=BLUE, label="PR-AUC")
axes[1].plot(epochs, history["val_roc_auc"], color=ORANGE, label="ROC-AUC")
axes[1].set_title("Validation ranking quality")
axes[1].set_xlabel("epoch"); axes[1].set_ylim(0, 1.02); axes[1].legend()

best = int(np.nanargmax(history["val_pr_auc"])) + 1
axes[1].axvline(best, color=GRID, linestyle="--", linewidth=1)
axes[1].annotate(f"best epoch {best}\nPR-AUC {max(history['val_pr_auc']):.3f}",
                 xy=(best, max(history["val_pr_auc"])), xytext=(6, -28),
                 textcoords="offset points", color=INK_2, fontsize=9)
fig.suptitle("Training history", x=0.02, ha="left", color=INK, fontsize=12)
plt.tight_layout(); plt.show()

## 7. EvaluationThe decision threshold is tuned on **validation only**, then frozen and applied once tothe test split. Test windows are enumerated deterministically every 15 days, so thisnumber is reproducible.

In [ ]:
results = evaluate_final(model, loaders, bundle, tcfg, device)
test_probs, test_targets = results["test_probs"], results["test_targets"]

In [ ]:
prec, rec, _ = precision_recall_curve(test_targets, test_probs)
fpr, tpr, _ = roc_curve(test_targets, test_probs)
m = results["test_at_tuned"]
base = test_targets.mean()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].plot(rec, prec, color=BLUE)
axes[0].axhline(base, color=GRID, linestyle="--", linewidth=1)
axes[0].annotate(f"random ranker = {base:.4f}", xy=(0.55, base), xytext=(0, 8),
                 textcoords="offset points", color=INK_2, fontsize=9)
axes[0].scatter([m["recall"]], [m["precision"]], s=60, color=ORANGE, zorder=3)
axes[0].annotate(f"tuned threshold\nP {m['precision']:.3f} / R {m['recall']:.3f}",
                 xy=(m["recall"], m["precision"]), xytext=(8, 8),
                 textcoords="offset points", color=INK_2, fontsize=9)
axes[0].set_title(f"Precision-Recall (PR-AUC {m['pr_auc']:.3f})")
axes[0].set_xlabel("recall"); axes[0].set_ylabel("precision"); axes[0].set_ylim(0, 1.02)

axes[1].plot(fpr, tpr, color=BLUE)
axes[1].plot([0, 1], [0, 1], color=GRID, linestyle="--", linewidth=1)
axes[1].set_title(f"ROC (ROC-AUC {m['roc_auc']:.3f})")
axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")

cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
axes[2].imshow(cm / cm.sum(axis=1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
axes[2].set_xticks([0, 1], ["predicted 0", "predicted 1"])
axes[2].set_yticks([0, 1], ["actual 0", "actual 1"])
axes[2].grid(False)
for i in range(2):
    for j in range(2):
        share = cm[i, j] / max(cm[i].sum(), 1)
        axes[2].text(j, i, f"{cm[i, j]:,}\n{100 * share:.1f}%", ha="center", va="center",
                     color="white" if share > 0.5 else INK, fontsize=10)
axes[2].set_title(f"Confusion @ threshold {m['threshold']:.3f}")

fig.suptitle("Test-set performance", x=0.02, ha="left", color=INK, fontsize=12)
plt.tight_layout(); plt.show()

## 8. Precision@K — the number an operator actually acts onNobody inspects every window. They work a queue: rank the fleet by risk, look at the topK. Precision@K is the share of that queue that really does contain a failure, and lift ishow many times better that is than picking windows at random.

In [ ]:
order = np.argsort(-test_probs)
sorted_labels = test_targets[order]
n_pos = int(test_targets.sum())
base = test_targets.mean()

rows = []
for k in [10, 25, 50, 100, 250, 500, max(n_pos, 1)]:
    k = min(k, len(sorted_labels))
    hits = int(sorted_labels[:k].sum())
    rows.append({"K": k, "hits": hits, "precision@K": hits / k,
                 "recall@K": hits / max(n_pos, 1), "lift": (hits / k) / base})
topk = pd.DataFrame(rows).drop_duplicates("K").sort_values("K").reset_index(drop=True)
display(topk.style.format({"precision@K": "{:.3f}", "recall@K": "{:.3f}", "lift": "{:.1f}x"}))

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(topk["K"].astype(str), topk["precision@K"], color=BLUE, width=0.6)
ax.axhline(base, color=GRID, linestyle="--", linewidth=1)
ax.annotate(f"random = {base:.4f}", xy=(0, base), xytext=(0, 6),
            textcoords="offset points", color=INK_2, fontsize=9)
for bar, p, lift in zip(bars, topk["precision@K"], topk["lift"]):
    ax.annotate(f"{p:.2f}\n({lift:.0f}x)",
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 4), textcoords="offset points",
                ha="center", color=INK_2, fontsize=9)
ax.set_title("Precision@K on the test split")
ax.set_xlabel("K (windows inspected, ranked by predicted risk)")
ax.set_ylabel("precision@K"); ax.set_ylim(0, 1.1)
plt.tight_layout(); plt.show()

## 9. Save the model and its preprocessing

In [ ]:
save_artifacts(model, bundle, tcfg, results, prefix=ARTIFACT_PREFIX)
print(json.dumps({k: v for k, v in results["test_at_tuned"].items()}, indent=2))

## 10. What this does and does not tell you**Reads on the numbers.** PR-AUC is the headline: with a positive rate well under 1%,ROC-AUC flatters every model (the huge true-negative pool dominates it) and accuracy ismeaningless. Compare PR-AUC against the base rate printed beside it, not against 1.0.**The strict label is hard by construction.** The daily-stats convention marks`failure == 1` on a drive's *last* reported day, so under `horizon_days = 0` exactly one window per failing drive ispositive — the one ending on its final day. Every earlier window of that same drive, evenone from the week before it died, is a negative. A model that fires a week early is*penalised*. Setting `horizon_days = 14` re-frames the task as "does this drive failwithin 14 days of the window's end", which is both easier and closer to what an operatorwants; it is a one-line change in cell 3.**Cohort, not fleet.** A single quarter of Seagate drives is not the whole population.Ranking metrics (PR-AUC, Precision@K) transfer; the absolute positive rate does not.**Next steps.** Compare the three architectures on identical splits; sweep`horizon_days`; check whether the `observed` mask channel earns its place by ablating it;and try `stride_days = 1` on the test split to score every possible window rather thanevery 15th.